# StyleStudio — Thí nghiệm định lượng Adaptive Fusion (Kaggle GPU)

Chạy sweep chọn `ρ` trên dev split, rồi thí nghiệm chính (adaptive vs fixed `end_fusion`), tính CLIP-T / CLIP-I(style) / layout-LPIPS. Dùng fork `namt9/StyleStudio` nhánh `adaptive-fusion` (đã có tín hiệu teacher-temporal + chunked fusion).

**Quota (T4, 50 bước, batch-2):** dev sweep ~18 run (~45 phút); main 5 điều kiện × 30 cặp = 150 run (~6–7h). **Resumable** — mỗi case ghi `.jpg`+`.json` ngay, chạy lại là bỏ qua case đã xong; đứt phiên 12h thì chạy tiếp.

## ⚙️ TRƯỚC KHI CHẠY
1. **Settings → Accelerator → `GPU T4 x2`**, **Internet → On** (cần xác thực SĐT tại `kaggle.com/settings`).
2. (Khuyến nghị) **Add-ons → Secrets → `HF_TOKEN`** (role Read) để tránh 403 khi tải model.
3. Chạy tuần tự các mục. Chạy **Mục 5 (dev sweep) + Mục 6 (chọn ρ)** trước; xem kết quả, **điền `CHOSEN_RHO`** ở Mục 7 rồi mới chạy main.

## 1. Kiểm tra môi trường

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Chua bat GPU! Settings -> Accelerator -> GPU T4 x2 (xac thuc SDT neu bi xam)'

In [ ]:
import os
# Lay HF_TOKEN tu Kaggle Secrets neu co (Add-ons -> Secrets -> HF_TOKEN).
# Khong bat buoc, nhung giup tranh 403 khi tai model an danh qua CDN Xet cua HF.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('Da lay HF_TOKEN tu Kaggle Secrets.')
except Exception as e:
    print('Khong co HF_TOKEN trong Secrets (bo qua neu chua can):', e)

## 2. Clone code (fork `namt9/StyleStudio`, nhánh `adaptive-fusion`)

In [ ]:
import os
REPO_DIR = '/kaggle/working/StyleStudio'
if not os.path.exists(REPO_DIR):
    !git clone -b adaptive-fusion https://github.com/namt9/StyleStudio.git {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -6
!ls

## 3. Cài dependencies (thêm `lpips`, `pandas` cho eval)

In [ ]:
!pip install -q \
  diffusers==0.25.1 \
  transformers==4.45.2 \
  tokenizers==0.20.1 \
  huggingface-hub==0.24.6 \
  accelerate peft safetensors einops omegaconf opencv-python pytest \
  lpips pandas
print('done')

## 4. Tải checkpoint + vá đường dẫn (như smoke test)

In [ ]:
import os, time, glob

# Tat backend "Xet" cua huggingface_hub - hay bi 403 khi tai an danh (khong token)
# tren cac repo lon nhu SDXL base. Phai dat TRUOC moi lan import/goi huggingface_hub.
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
!pip uninstall -y hf_xet -q

from huggingface_hub import hf_hub_download, snapshot_download, login

HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_TOKEN:
    login(token=HF_TOKEN)
    print('Da dang nhap HuggingFace bang token.')
else:
    print('CANH BAO: chua co HF_TOKEN. Neu gap loi 403 khi tai model, tao token (role Read) '
          'tai https://huggingface.co/settings/tokens roi them vao Kaggle Secrets (xem cell dau).')

# Bo qua anh preview/docs/openvino khong can thiet cho pipeline PyTorch
# (giam so file phai tai, giam rui ro trung blob dang loi tren CDN Xet)
SKIP_ASSETS = ['*.png', '*.jpg', '*.jpeg', '*.gif', '*.md', '*.PNG', '*.JPG',
               '*openvino*', 'onnx/*', '*.onnx']

def snapshot_download_retry(repo_id, retries=10, delay=20, **kw):
    """max_workers=1 (tuan tu) de tranh CDN rate-limit/403 khi tai song song.
    Retry dai + backoff tang dan vi loi 403 tren xet-bridge hien la su co ha tang
    HuggingFace tren GCP, chap chon theo edge server (xem link trong markdown tren)."""
    for attempt in range(1, retries + 1):
        try:
            return snapshot_download(repo_id, max_workers=1, **kw)
        except Exception as e:
            wait = delay * attempt
            print(f'  [{repo_id}] lan {attempt}/{retries} loi: {e}')
            if attempt == retries:
                raise
            print(f'  cho {wait}s roi thu lai...')
            time.sleep(wait)

# --- SDXL base: uu tien Kaggle Model (khong qua CDN Xet dang loi), fallback ve HuggingFace ---
BASE_MODEL_LOCAL = None
kaggle_model_hits = glob.glob('/kaggle/input/**/model_index.json', recursive=True)
if kaggle_model_hits:
    BASE_MODEL_LOCAL = os.path.dirname(kaggle_model_hits[0])
    print('Dung SDXL base tu Kaggle Model:', BASE_MODEL_LOCAL)
else:
    print('Khong thay Kaggle Model SDXL base da attach -> tai qua HuggingFace (co the cham/403).')
    print('Neu muon tranh 403: Add Input -> Models -> stabilityai/stable-diffusion-xl '
          '(Framework PyTorch, Variation base-1-0) roi chay lai cell nay.')
    snapshot_download_retry('stabilityai/stable-diffusion-xl-base-1.0', ignore_patterns=SKIP_ASSETS)
    print('Da cache SDXL base qua HuggingFace.')

snapshot_download_retry('madebyollin/sdxl-vae-fp16-fix', ignore_patterns=SKIP_ASSETS)
print('Da cache VAE.')

ie_root = snapshot_download_retry('h94/IP-Adapter', allow_patterns=['sdxl_models/image_encoder/*'])
IMAGE_ENCODER_LOCAL = os.path.join(ie_root, 'sdxl_models', 'image_encoder')
print('image_encoder:', IMAGE_ENCODER_LOCAL)

CSGO_LOCAL = hf_hub_download('InstantX/CSGO', 'csgo_4_32.bin')
print('csgo ckpt   :', CSGO_LOCAL)

def patch_paths(path):
    with open(path) as f:
        s = f.read()
    s = s.replace("os.environ['HF_ENDPOINT']='https://hf-mirror.com'",
                  "# HF mirror removed for Kaggle")
    if BASE_MODEL_LOCAL:
        s = s.replace('base_model_path = "stabilityai/stable-diffusion-xl-base-1.0"',
                      f'base_model_path = "{BASE_MODEL_LOCAL}"')
    s = s.replace('image_encoder_path = "h94/IP-Adapter/sdxl_models/image_encoder"',
                  f'image_encoder_path = "{IMAGE_ENCODER_LOCAL}"')
    s = s.replace("csgo_ckpt ='InstantX/CSGO/csgo_4_32.bin'",
                  f"csgo_ckpt ='{CSGO_LOCAL}'")
    with open(path, 'w') as f:
        f.write(s)
    print('patched:', path)

for p in ['infer_StyleStudio.py', 'infer_StyleStudio_layout_stability.py']:
    if os.path.exists(p):
        patch_paths(p)

## 4b. Vá `run_experiments.py` dùng đường dẫn local

`run_experiments.py` hardcode base model theo HF id; nếu bạn attach Kaggle Model thì trỏ về local để tránh 403. `image_encoder`/`csgo` truyền qua tham số (`COMMON`).

In [ ]:
import os
# base model: neu co Kaggle Model (BASE_MODEL_LOCAL) thi tro run_experiments ve local
_rp = 'experiments/run_experiments.py'
_s = open(_rp).read()
if BASE_MODEL_LOCAL and BASE_MODEL_LOCAL not in _s:
    _s = _s.replace('"stabilityai/stable-diffusion-xl-base-1.0",', f'"{BASE_MODEL_LOCAL}",')
    open(_rp, 'w').write(_s)
    print('patched run_experiments base ->', BASE_MODEL_LOCAL)
else:
    print('run_experiments base: dung HF cache (khong co Kaggle Model hoac da patch).')

# tham so chung cho moi lenh run_experiments (image_encoder + csgo local, tu MUC 4)
COMMON = f'--image_encoder_path "{IMAGE_ENCODER_LOCAL}" --csgo_ckpt "{CSGO_LOCAL}"'
ENV = 'HF_HUB_DISABLE_XET=1 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True'
print('COMMON =', COMMON)

## 5. Sanity: unit test (nhanh)

In [ ]:
!python -m pytest tests/ -q

## 6. Dev ρ-sweep — chọn `ρ`

Chạy `adaptive` với ρ ∈ {0.5, 0.6, 0.7} trên **2 dev prompt × 3 style** (50 bước). Tín hiệu teacher-temporal chạm đáy ~0.5 nên ρ nằm khoảng này (0.1–0.3 của thiết kế cũ không dùng được). Kết quả vào `experiments/outputs_dev/`.

In [ ]:
import subprocess
for rho in [0.5, 0.6, 0.7]:
    tag = f'rho{int(round(rho*100)):02d}'
    cmd = (f'{ENV} python experiments/run_experiments.py '
           f'--condition adaptive --rho {rho} --tag {tag} --end_fusion_max 30 --num_steps 50 '
           f'--prompts experiments/prompts_dev.txt --styles_dir experiments/styles '
           f'--out_root experiments/outputs_dev {COMMON}')
    print('\n>>', cmd)
    subprocess.run(cmd, shell=True, check=True)
print('\n=== dev sweep xong ===')

## 6b. Tính metrics dev + chọn `ρ`

In [ ]:
import subprocess, pandas as pd
subprocess.run(f'{ENV} python experiments/eval_metrics.py '
               f'--out_root experiments/outputs_dev --styles_dir experiments/styles '
               f'--results_dir experiments/results_dev', shell=True, check=True)
df = pd.read_csv('experiments/results_dev/results.csv')
g = (df.groupby('condition')
       .agg(clip_t=('clip_t','mean'), clip_i_style=('clip_i_style','mean'),
            stop_step=('stop_step','mean'), sec=('elapsed_sec','mean'))
       .round(4).sort_index())
print(g)
print('\nChon rho: uu tien clip_t (bam prompt) & clip_i_style (bam style) cao,')
print('stop_step hop ly (khong dung qua som lam mat layout, khong qua muon = it tiet kiem).')
print('Ghi gia tri rho da chon vao CHOSEN_RHO o MUC 7.')

## 7. Thí nghiệm chính (adaptive vs fixed)

**Điền `CHOSEN_RHO`** từ Mục 6b. Chạy 5 điều kiện trên **10 test prompt × 3 style** (50 bước). Resumable — cứ chạy lại nếu đứt phiên. Kết quả vào `experiments/outputs/`.

> ~150 run (~6–7h GPU). Có thể chạy từng điều kiện (sửa `CONDS`) qua nhiều phiên.

In [ ]:
CHOSEN_RHO = 0.6      # <-- SUA theo Muc 6b

import subprocess
def run_cond(cond, extra=''):
    cmd = (f'{ENV} python experiments/run_experiments.py --condition {cond} {extra} '
           f'--num_steps 50 --prompts experiments/prompts_test.txt --styles_dir experiments/styles '
           f'--out_root experiments/outputs {COMMON}')
    print('\n>>', cmd)
    subprocess.run(cmd, shell=True, check=True)

for c in ['fixed5', 'fixed10', 'fixed20', 'fixed30']:
    run_cond(c)
run_cond('adaptive', f'--rho {CHOSEN_RHO} --end_fusion_max 30')   # out dir = "adaptive"
print('\n=== main xong (hoac dung phan da co, chay lai de tiep) ===')

## 8. Tính metrics + tổng hợp bảng/figure

In [ ]:
import subprocess
subprocess.run(f'{ENV} python experiments/eval_metrics.py '
               f'--out_root experiments/outputs --styles_dir experiments/styles '
               f'--results_dir experiments/results', shell=True, check=True)
subprocess.run('python experiments/analyze_results.py '
               '--results_dir experiments/results --out_root experiments/outputs',
               shell=True, check=True)

## 9. Xem kết quả

In [ ]:
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

print('=== summary_table.csv ==='); print(pd.read_csv('experiments/results/summary_table.csv'))
print('\n=== winners.csv (fixed nao thang moi case + gap cua adaptive) ===')
print(pd.read_csv('experiments/results/winners.csv'))

for fig in ['hist_stop_steps.png', 'r_curves.png']:
    p = f'experiments/results/{fig}'
    try:
        plt.figure(figsize=(7,4)); plt.imshow(Image.open(p)); plt.axis('off'); plt.title(fig); plt.show()
    except FileNotFoundError:
        print('chua co', p)